In [5]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Load the updated JSON data from a file
with open('domainpages.json', 'r') as file:
    pages = json.load(file)

# Load the project data from pages.json
with open('pages.json', 'r') as file:
    projects = json.load(file)

# Cloudflare API credentials
account_id = os.getenv('CLOUDFLARE_ACCOUNT_ID')
api_token = os.getenv('CLOUDFLARE_API_TOKEN')

headers = {
    'Authorization': f'Bearer {api_token}',
    'Content-Type': 'application/json'
}

def remove_domain_from_project(project_name, domain):
    # project_id = get_project_id(project_name)
    project_id = project_name
    if project_id:
        print(f"Removing domain {domain} from project {project_name}")
        url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/pages/projects/{project_id}/domains/{domain}"
        response = requests.delete(url, headers=headers)
        if response.status_code == 200:
            print(f"Successfully removed domain {domain} from project {project_name}")
        else:
            print(f"Failed to remove domain {domain} from project {project_name}: {response.text}")
    else:
        print(f"Project ID not found for {project_name}")

def add_domain_to_project(project_name, domain):
    # project_id = get_project_id(project_name)
    project_id = project_name
    if project_id:
        print(f"Adding domain {domain} to project {project_name}")
        url = f"https://api.cloudflare.com/client/v4/accounts/{account_id}/pages/projects/{project_id}/domains"
        data = {
            "name": domain
        }
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 200:
            print(f"Successfully added domain {domain} to project {project_name}")
        else:
            print(f"Failed to add domain {domain} to project {project_name}: {response.text}")
    else:
        print(f"Project ID not found for {project_name}")

def get_project_id(project_name):
    for project in projects:
        if project['name'] == project_name:
            return project['id']
    return None

# Process pages to remove and add domains
for page in pages:
    for domain in page['domains']:

        if 'allwomenstalk.com' in domain:
            # Identify the original and target project names
            original_project_name = 'allwomenstalk-' + page['name'].split('-')[0]
            target_project_name = page['name']

            # Remove domain from the original project
            remove_domain_from_project(original_project_name, domain)

            # Add domain to the target project
            add_domain_to_project(target_project_name, domain)
            page['domain']=domain
            break

# Save the updated JSON data to a file
with open('routes.json', 'w') as file:
    json.dump(pages, file, indent=2)